# Reference · Exploring the Ames data

**This notebook is a tool, not a reading.** Almost every cell has something in it you
are meant to change — a column name, a threshold, a list. Change it, re-run the cell,
see what happens. You will learn more in ten minutes of that than from reading it
straight through.

Nothing here is assessed. It exists because you will be handed this dataset repeatedly
and you should know what is actually in it.

## ⚠ First, a rule that is not optional in this course

**Split before you explore, and explore the training half.**

This feels like bureaucracy and it is not. Every decision you make after looking at the
data — which columns to keep, which outlier to drop, which transformation to apply — is
a choice fitted to what you saw. If you saw all 2,930 houses, those choices were fitted
to all 2,930 houses, and the "held-out" quarter was not held out from *you*.

The effect is usually small for looking at a histogram, and it is not small at all for
"I dropped the six weird houses" or "I picked the eight columns that correlated best."
Those are model decisions wearing exploration's clothes. ESL §7.10.2 is three pages on
exactly this, and it is Thursday's whole meeting.

So: the test set below is created and then **never touched again** in this notebook.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from stat764 import load

pd.set_option("display.width", 160)

ames = load("ames.csv")
train, test = train_test_split(ames, test_size=0.25, random_state=764)

print(f"all      {ames.shape[0]:>5} rows x {ames.shape[1]} columns")
print(f"train    {train.shape[0]:>5}   <- everything below uses only this")
print(f"test     {test.shape[0]:>5}   <- set aside, not looked at")

## 1. What is in here at all

Eighty-two columns is too many to read one at a time, so start with the shape of the
collection rather than the contents.

In [ ]:
print("column types:")
print(train.dtypes.value_counts().to_string())

print(f"\nthe outcome — SalePrice:")
print(f"  {train.SalePrice.min():>8,}  min")
print(f"  {train.SalePrice.median():>8,.0f}  median")
print(f"  {train.SalePrice.max():>8,}  max")
print(f"  skew {train.SalePrice.skew():.2f}   (0 would be symmetric)")

**Try this:** that skew is why people often model `log(SalePrice)` instead. Run
`np.log(train.SalePrice).skew()` in a new cell and see what it becomes.

## 2. One column at a time — the workhorse

`look()` handles numeric and categorical columns differently, because they need
different questions asked of them.

**Change the name in the last line and re-run.** Any of the 82 columns works.

In [ ]:
def look(col, data=train, top=8):
    """Summarise one column and its relationship to SalePrice."""
    s = data[col]
    missing = s.isna().sum()
    print(f"{col}   dtype {s.dtype}   missing {missing} of {len(s)}"
          f" ({missing / len(s):.1%})")

    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
        print(f"  min {s.min():,.1f}   median {s.median():,.1f}   max {s.max():,.1f}"
              f"   skew {s.skew():.2f}")
        print(f"  correlation with SalePrice: {s.corr(data.SalePrice):.3f}")
    else:
        vc = s.value_counts(dropna=False).head(top)
        med = data.groupby(s.astype("object"), dropna=False).SalePrice.median()
        print(f"  {s.nunique()} distinct values. Top {min(top, len(vc))}:")
        print(f"    {'value':<16}{'n':>6}{'median price':>15}")
        for v, n in vc.items():
            key = v if not pd.isna(v) else np.nan
            m = med.get(key, np.nan)
            print(f"    {str(v):<16}{n:>6}{m:>15,.0f}" if pd.notna(m)
                  else f"    {str(v):<16}{n:>6}{'—':>15}")


look("Gr_Liv_Area")      # <- CHANGE ME. try "Neighborhood", "Overall_Qual", "Pool_QC"

## 3. Missing values, and what they mean

Missing is not one thing. Sometimes a measurement failed; sometimes the feature does
not exist for that house. Those need opposite treatment, and only you can tell which
is which — the dataframe cannot.

In [ ]:
miss = train.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print(f"{len(miss)} of {train.shape[1]} columns have missing values\n")
print(f"  {'column':<18}{'missing':>9}{'%':>8}")
for c, n in miss.head(12).items():
    print(f"  {c:<18}{n:>9}{n / len(train):>8.1%}")

Look at the top of that list: `Pool_QC`, `Misc_Feature`, `Alley`, `Fence`. These are
not failed measurements. **The house has no pool.** Filling them with a median or a
most-frequent value would invent a pool.

The honest move is usually to make "missing" its own category, so the model can learn
that not having a pool is informative. That is what
`SimpleImputer(strategy="constant", fill_value="Missing")` is for.

In [ ]:
# Does "has no pool" carry information about price? Check before assuming either way.
for col in ["Pool_QC", "Fireplace_Qu", "Garage_Type", "Alley"]:
    has, hasnt = train[train[col].notna()].SalePrice, train[train[col].isna()].SalePrice
    if len(has) and len(hasnt):
        print(f"  {col:<16} present n={len(has):>4} median {has.median():>8,.0f}   "
              f"absent n={len(hasnt):>4} median {hasnt.median():>8,.0f}")

**Try this:** add `"Fence"` or `"Misc_Feature"` to that list. Is the gap always in the
direction you expected?

## 4. The trap: integers that are not quantities

`Neighborhood` is obviously categorical because it is text. `MS_SubClass` is a dwelling
**type code** stored as an integer, and nothing about its dtype says so. Standardising
it, or letting a linear model treat 20 and 60 as three-times-apart, is nonsense.

Any rule that routes columns by dtype gets these wrong. There is no substitute for
reading the data dictionary.

In [ ]:
suspects = [c for c in train.select_dtypes("number").columns
            if c != "SalePrice" and train[c].nunique() <= 16]

print(f"{len(suspects)} numeric columns with 16 or fewer distinct values:\n")
print(f"  {'column':<18}{'distinct':>9}   values")
for c in sorted(suspects, key=lambda c: train[c].nunique()):
    vals = sorted(train[c].dropna().unique())
    shown = ", ".join(f"{v:g}" for v in vals[:7]) + (" ..." if len(vals) > 7 else "")
    print(f"  {c:<18}{train[c].nunique():>9}   {shown}")

Sort that list into three piles yourself. Some are genuine counts (`Fireplaces`), some
are ordered ratings you might reasonably treat either way (`Overall_Qual`), and some
are codes pretending to be numbers (`MS_SubClass`, `Mo_Sold`).

**The pile something belongs in is a modelling decision, not a fact about the file.**

## 5. What relates to price

Correlation only sees straight lines, so treat this as a place to start looking rather
than an answer.

In [ ]:
num = train.select_dtypes("number").drop(columns="SalePrice")
corr = num.corrwith(train.SalePrice).sort_values(ascending=False)
print("strongest positive:"); print(corr.head(8).round(3).to_string())
print("\nstrongest negative:"); print(corr.tail(4).round(3).to_string())
print("\n  ^ one of those four is not a property of the house at all. Next section.")

In [ ]:
def versus(col, data=train, log_price=False):
    """Plot one column against SalePrice, picking the right kind of plot."""
    price = np.log10(data.SalePrice) if log_price else data.SalePrice
    label = "log10(SalePrice)" if log_price else "SalePrice"
    fig, ax = plt.subplots(figsize=(7, 4.5))

    if pd.api.types.is_numeric_dtype(data[col]) and data[col].nunique() > 16:
        ax.scatter(data[col], price, s=8, alpha=0.3)
        ax.set_xlabel(col)
    else:
        groups = data.groupby(data[col].astype("object"), dropna=False)
        order = groups.SalePrice.median().sort_values().index
        ax.boxplot([price[data[col].astype("object") == g] for g in order],
                   tick_labels=[str(g)[:10] for g in order])
        ax.tick_params(axis="x", rotation=90)
        ax.set_xlabel(f"{col}  (ordered by median price)")
    ax.set_ylabel(label)
    ax.set_title(f"{col} vs {label}")
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


versus("Gr_Liv_Area")    # <- CHANGE ME. try "Neighborhood", "Overall_Qual", "Year_Built"

**Try this:** run `versus("Gr_Liv_Area", log_price=True)`. The cloud straightens out,
which is another way of seeing the skew from section 1.

**And this:** `versus("Neighborhood")` — 28 boxes, sorted by median. That plot is worth
more than the correlation table for a categorical.

## 6. Look again at that negative list

`PID` is in it, at −0.25. `PID` is the **parcel identification number** — a serial
number the assessor's office assigns to a piece of land. It is not a fact about the
house. It should carry no information about price at all.

It carries plenty. Find out why before reading on.

In [ ]:
ids = [c for c in train.columns if train[c].is_unique]
print(f"columns that are unique for every row (i.e. identifiers): {ids}\n")

prefix = train.PID.astype(str).str[:3]
g = (train.assign(prefix=prefix)
          .groupby("prefix")
          .agg(n=("SalePrice", "size"),
               median_price=("SalePrice", "median"),
               most_common_neighborhood=("Neighborhood", lambda s: s.mode().iloc[0])))
print("first three digits of PID, where there are at least 40 sales:")
print(g[g.n >= 40].sort_values("median_price").to_string())

The prefix **is** the neighbourhood. `902` is OldTown at a median of about $119k; `528`
is Northridge Heights at about $253k. The parcel numbering encodes geography, geography
drives price, and so a meaningless serial number correlates with the outcome.

⚠ **This is the shape of most leakage.** Not a column called `price_next_year`, but a
column that works *for a reason you did not intend and cannot defend.* A model using
`PID` has learned the Story County parcel numbering scheme. Renumber the parcels, or
take the model to a different county, and it has learned nothing.

And notice how easy it is to include by accident: `train.select_dtypes("number")`
sweeps `Order` and `PID` straight in, and they will improve your R-squared. The "every
numeric column" row in the Day 2 reference had both of them in it.

**Drop identifiers before you model.** Then ask, of every remaining column, whether it
would exist — and mean the same thing — for a house you have not sold yet. Thursday is
ninety minutes of that question.

## 7. Rare categories, and why they break things

A level with two houses in it is a problem waiting for a random split.

In [ ]:
vc = train.Neighborhood.value_counts()
rare = vc[vc < 10]
print(f"{len(vc)} neighborhoods in the training half; {len(rare)} have fewer than 10 sales:")
print(rare.to_string() if len(rare) else "  none")

print("\n  A neighborhood absent from a training fold but present in the test fold will")
print("  make OneHotEncoder raise — unless you passed handle_unknown='ignore'.")
print("  Over the 50 splits in Meeting 3 this is close to a certainty, not an edge case.")

## Try these

Open a new cell for each. There are no answers at the bottom; the point is the looking.

| | |
|---|---|
| **1** | Run `look()` on five columns you have never heard of. Which one surprised you? |
| **2** | `versus("Overall_Qual")` and `versus("Overall_Cond")`. One of them relates to price far more cleanly. Why might that be? |
| **3** | Find the three most expensive houses in `train`. Are they unusual in a way a model could learn, or just unusual? |
| **4** | `train.groupby("Yr_Sold").SalePrice.median()` — these are 2006–2010. Does the crash show up? |
| **5** | Pick any two columns you expect to be redundant and check `train[[a, b]].corr()`. Redundancy matters in Week 4. |

⚠ And the one that is not optional: **everything above ran on `train`.** If you now go
and check whether any of it also holds in `test`, you have spent your test set. Do not.

---

## Also in this neighborhood

📗 **The data dictionary.** De Cock (2011), *Journal of Statistics Education* 19(3) —
the paper that released this dataset, with every column defined. Free, and short.

📗 **`ref_sklearn_structures.ipynb`** — what `Pipeline` and `ColumnTransformer` are
doing, if the preprocessing in the meeting notebooks still feels like copied syntax.

🚫 **Automated EDA packages** (`pandas-profiling`, `sweetviz` and friends). They
produce a hundred pages, none of which is a decision. Worse, they run on everything you
hand them — which, if you hand them the whole file, is the mistake at the top of this
notebook, automated.